In [1]:
import torch, platform

print("Python           :", platform.python_version())
print("PyTorch          :", torch.__version__)
print("CUDA disponible  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU              :", torch.cuda.get_device_name(0))
    print("Mémoire (Go)     :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("⚠️  AUCUN GPU — activez : Exécution > Modifier le type d'exécution > GPU")

Python           : 3.12.13
PyTorch          : 2.11.0+cu128
CUDA disponible  : True
GPU              : Tesla T4
Mémoire (Go)     : 15.64


In [2]:
!nvidia-smi

Mon Aug 17 11:30:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/plant-disease')
for sub in ['dataset', 'models', 'results', 'results/figures']:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("Arborescence Drive prête :")
for p in sorted(DRIVE_ROOT.rglob('*')):
    print("  ", p.relative_to(DRIVE_ROOT))

Arborescence Drive prête :
   dataset
   models
   results
   results/figures


In [5]:
!pip install -q timm grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 85.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
import timm
print("timm :", timm.__version__)
print("Modèles ViT-Tiny disponibles :", timm.list_models('vit_tiny*', pretrained=True))

timm : 1.0.28
Modèles ViT-Tiny disponibles : ['vit_tiny_patch16_224.augreg_in21k', 'vit_tiny_patch16_224.augreg_in21k_ft_in1k', 'vit_tiny_patch16_384.augreg_in21k_ft_in1k', 'vit_tiny_patch16_dinov3_qkvb.eupe_lvd1689m', 'vit_tiny_r_s16_p8_224.augreg_in21k', 'vit_tiny_r_s16_p8_224.augreg_in21k_ft_in1k', 'vit_tiny_r_s16_p8_384.augreg_in21k_ft_in1k']


In [7]:
# Option A — recommandée : cloner votre dépôt (après un premier push sur GitHub)
!git clone https://github.com/kabdoullah/plant-disease-classification.git /content/project
%cd /content/project

Cloning into '/content/project'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 18 (delta 4), reused 17 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 7.10 KiB | 3.55 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/project


In [8]:
import sys
sys.path.insert(0, '/content/project')

from src import config
from src.utils import set_seed, get_device

set_seed(config.SEED)
device = get_device()
print("Device      :", device)
print("Classes     :", config.NUM_CLASSES)
print("Dossier poids:", config.MODELS_DIR)

Device      : cuda
Classes     : 5
Dossier poids: /content/drive/MyDrive/plant-disease/models


In [9]:
import torch.nn as nn
import torch.optim as optim
from src.utils import save_checkpoint, load_checkpoint, file_size_mb
from src.metadata import build_metadata

# Modèle jouet, uniquement pour tester la mécanique de sauvegarde
dummy = nn.Linear(10, config.NUM_CLASSES)
opt   = optim.Adam(dummy.parameters(), lr=1e-3)

ckpt_path = config.MODELS_DIR / "_test_checkpoint.pt"
save_checkpoint(
    ckpt_path,
    model=dummy,
    optimizer=opt,
    epoch=0,
    history={"train_loss": [], "val_acc": []},
    metadata=build_metadata("dummy"),
)
print(f"Écrit  : {ckpt_path}  ({file_size_mb(ckpt_path):.3f} Mo)")

# Relecture
restored = nn.Linear(10, config.NUM_CLASSES)
ck = load_checkpoint(ckpt_path, model=restored)
print("Relu   :", ck["metadata"]["model_name"], "| classes :", ck["metadata"]["classes"])

ckpt_path.unlink()
print("Test OK, fichier temporaire supprimé.")

Écrit  : /content/drive/MyDrive/plant-disease/models/_test_checkpoint.pt  (0.003 Mo)
Relu   : dummy | classes : ['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Septoria_leaf_spot', 'Tomato___healthy']
Test OK, fichier temporaire supprimé.
